# Volume 5, Chapter 1: Mertens's theorems (Python companion)

Exercises first, solutions at the end. We sieve the primes up to $10^7$, watch Chebyshev's functions and Mertens's three theorems converge, compute Mertens's constant from values of $\zeta$, meet the mysterious factor $2e^{-\gamma}$, and see how Mertens's third theorem predicts the speed-up from trial division in RSA key generation. Requirements: numpy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★) Three theorems, one table.** Sieve the primes up to $10^7$. For $x=10^2,\dots,10^7$, tabulate $\theta(x)/x$, $R(x)=\sum_{p\le x}\frac{\log p}p-\log x$ (Theorem 1.3), $\sum_{p\le x}\frac1p-\log\log x$ (Theorem 1.5) and $\log x\prod_{p\le x}(1-\frac1p)$ (Theorem 1.6). Which column converges fastest? Then compare $\psi$ and $\theta$ at $10^7$, and check the lower bound $\psi(x)\ge x\log2-3\log x-3$ of Exercise 1.3 for all $x\le10^5$.

**P1.2 (★★) Mertens's constant from $\zeta$.** A sum over primes converging like $\sum_p p^{-2}$ is too slow for many digits. Use Exercise 1.2 and the identity $\sum_p\bigl(\log(1-\frac1p)+\frac1p\bigr)=\sum_{k\ge2}\frac{\mu(k)}k\log\zeta(k)$ (prove it: expand $\log(1-\frac1p)$ and use Möbius inversion on $\log\zeta(s)=\sum_p\sum_m\frac{p^{-ms}}m$) to compute $M$ and $H$ to twenty digits, and check $M+H=\gamma$ (Lemma 1.7). Also compute the conjectured limit $-\gamma-\sum_p\frac{\log p}{p(p-1)}$ of $R(x)$, and the constants $2e^{-\gamma}$ and $\frac{6e^\gamma}{\pi^2}$.

**P1.3 (★) The factor $2e^{-\gamma}$.** Compare the true density $\pi(x)/x$ of primes with the "independence" prediction $\prod_{p\le\sqrt x}(1-\frac1p)$ for $x=10^3,\dots,10^7$. Then check Exercise 1.5 numerically.

**P1.4 (★★) CS application: sieving before Miller–Rabin.** To generate a random 1024-bit prime, one draws random odd numbers and tests them with Miller–Rabin, which costs a modular exponentiation. Cheap trial division by the odd primes up to $B$ first discards most candidates. By Exercise 1.1 and Theorem 1.6, what fraction survives? Measure it on $100\,000$ random odd 1024-bit numbers, then measure the number of Miller–Rabin calls and the time per prime for $B=0,100,2000,20\,000$.

## Solutions

### P1.1

In [ ]:
import math, random, time
import numpy as np
import mpmath as mp

def prime_sieve(n):
    """Boolean array s with s[k] True exactly when k is prime, for 0 <= k <= n (sieve of Eratosthenes)."""
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for p in range(2, math.isqrt(n) + 1):
        if s[p]:
            s[p * p::p] = False
    return s

X = 10**7
primes = np.flatnonzero(prime_sieve(X))
logp = np.log(primes)
print(f"{len(primes)} primes below {X:.0e}, the largest is {primes[-1]}")

gamma = float(mp.euler)
M_TRUE = 0.2614972128476428   # Meissel-Mertens constant, checked in S2
print("    x        theta/x   R(x)=sum log p/p - log x   sum 1/p - loglog x   log x * prod(1-1/p)")
cum_theta = np.cumsum(logp)
cum_lp = np.cumsum(logp / primes)
cum_inv = np.cumsum(1.0 / primes)
cum_log1m = np.cumsum(np.log1p(-1.0 / primes))
for e in range(2, 8):
    x = 10**e
    i = np.searchsorted(primes, x, side="right") - 1
    print(f"  1e{e}    {cum_theta[i] / x:.4f}     {cum_lp[i] - math.log(x):+.4f}"
          f"                  {cum_inv[i] - math.log(math.log(x)):.6f}            {math.log(x) * math.exp(cum_log1m[i]):.6f}")
print(f"limits:   1        -1.3326 (see S2)            M = {M_TRUE:.6f}            e^-gamma = {math.exp(-gamma):.6f}")

In [ ]:
psi_minus_theta = 0.0
k = 2
while 2**k <= X:
    lim = int(round(X ** (1 / k)))
    while lim**k > X: lim -= 1
    while (lim + 1)**k <= X: lim += 1
    psi_minus_theta += logp[: np.searchsorted(primes, lim, side="right")].sum()
    k += 1
psi = cum_theta[-1] + psi_minus_theta
print(f"theta(1e7)/1e7 = {cum_theta[-1] / X:.6f}   psi(1e7)/1e7 = {psi / X:.6f}   (psi - theta)/sqrt(x) = {psi_minus_theta / math.sqrt(X):.4f}")

# Chebyshev's lower bound (Exercise): psi(x) >= x log 2 - 3 log x - 3, checked for every integer 2 <= x <= 10^5
N = 10**5
lam = np.zeros(N + 1)
for p in primes[primes <= N]:
    q = p
    while q <= N:
        lam[q] = math.log(p)
        q *= p
psi_n = np.cumsum(lam)
xs = np.arange(N + 1)
slack = psi_n[2:] - (xs[2:] * math.log(2) - 3 * np.log(xs[2:]) - 3)
print("min over 2 <= x <= 1e5 of psi(x) - (x log 2 - 3 log x - 3):", round(slack.min(), 3), "at x =", 2 + int(slack.argmin()))

$\theta(x)/x$ creeps up to $1$ (the prime number theorem, Chapter 5), but at $10^7$ it is still $0.9995$. $R(x)$ settles near $-1.3326$, the limit computed in P1.2. The second and third columns converge fastest, with errors that look like $O(1/\log x)$ but are in fact much smaller; Theorems 1.5 and 1.6 are proved with $O(1/\log x)$ only. At $10^7$, Mertens's second difference is $0.261507$, within $10^{-5}$ of $M$, and $\log x\prod(1-\frac1p)=0.561454$ against $e^{-\gamma}=0.561459$. The prime powers add about $\sqrt x$ to $\psi$: $(\psi-\theta)(x)\approx\theta(\sqrt x)\approx\sqrt x$. Chebyshev's lower bound has a comfortable margin, smallest at $x=2$.

### P1.2

In [ ]:
mp.mp.dps = 30
# M = gamma + sum_p (log(1-1/p) + 1/p), and sum_p log(1-1/p) + 1/p = sum_{k>=2} mu(k) log zeta(k) / k.
def mobius(n):
    res, m, p = 1, n, 2
    while p * p <= m:
        if m % p == 0:
            m //= p
            if m % p == 0:
                return 0
            res = -res
        p += 1
    return -res if m > 1 else res

S = mp.mpf(0)
for k in range(2, 120):
    mu = mobius(k)
    if mu:
        S += mu * mp.log(mp.zeta(k)) / k
M = mp.euler + S
H = -S
print("M  =", mp.nstr(M, 20), "  (from zeta values)")
print("H  =", mp.nstr(H, 20), "  M + H - gamma =", mp.nstr(M + H - mp.euler, 3))
print("direct partial sum of h(p) up to 1e7:", f"{np.sum(-np.log1p(-1.0 / primes) - 1.0 / primes):.10f}  (tail < 1e-7)")

# the limit of R(x): -gamma - sum_p log p / (p (p - 1)); the sum converges fast, so a prime sum plus a tail bound suffices
B = float(np.sum(logp / (primes * (primes - 1.0))))
print(f"sum log p/(p(p-1)) up to 1e7 = {B:.10f}   ->   -gamma - B = {-gamma - B:.6f}")

print("2 e^-gamma =", mp.nstr(2 * mp.exp(-mp.euler), 8), "  6 e^gamma / pi^2 =", mp.nstr(6 * mp.exp(mp.euler) / mp.pi**2, 8))

Writing $\log(1-\frac1p)+\frac1p=-\sum_{m\ge2}\frac{p^{-m}}m$ and summing over $p$ gives $-\sum_{m\ge2}\frac{P(m)}m$, with $P(s)=\sum_pp^{-s}$; since $\log\zeta(s)=\sum_{m\ge1}\frac{P(ms)}m$, Möbius inversion gives $P(s)=\sum_{k\ge1}\frac{\mu(k)}k\log\zeta(ks)$, and regrouping yields the identity (all series converge absolutely for $s\ge2$). The terms decay like $2^{-k}/k$, so $120$ terms give far more than twenty digits: $M=0.26149721284764278376$, $H=0.31571845205389007685$, and $M+H-\gamma$ vanishes to working precision. The direct prime sum up to $10^7$ agrees with $H$ to $3\cdot10^{-9}$, within its tail bound $\frac1{x-1}$ of Theorem 1.6's proof. The limit of $R(x)$ is $-1.332582$, consistent with the table in P1.1. Finally $2e^{-\gamma}=1.1229$ and $\frac{6e^\gamma}{\pi^2}=1.0828$.

### P1.3

In [ ]:
# The factor 2 e^-gamma: the "independence" prediction prod_{p <= sqrt x}(1 - 1/p) against the true density pi(x)/x.
pi_of = lambda x: np.searchsorted(primes, x, side="right")
print("    x      pi(x)/x       prod_{p<=sqrt x}(1-1/p)    ratio")
for e in range(3, 8):
    x = 10**e
    i = pi_of(math.isqrt(x)) - 1
    prod = math.exp(cum_log1m[i])
    print(f"  1e{e}   {pi_of(x) / x:.6f}      {prod:.6f}                  {prod / (pi_of(x) / x):.4f}")
print(f"limit: 2 e^-gamma = {2 * math.exp(-gamma):.4f}")

# Plus product (Exercise): prod (1 + 1/p) / log x
cum_log1p = np.cumsum(np.log1p(1.0 / primes))
for e in (3, 5, 7):
    i = pi_of(10**e) - 1
    print(f"x = 1e{e}: prod_(p<=x)(1+1/p) / log x = {math.exp(cum_log1p[i]) / math.log(10**e):.5f}")

The ratio drifts upward through $1$ around $x=10^5$ and is still only $1.047$ at $10^7$. Theorem 1.6 together with the prime number theorem says it tends to $2e^{-\gamma}=1.1229$, slowly, since both sides have errors of relative size $\frac1{\log x}$. The prediction underestimates small primes because $\prod_{p\le\sqrt x}$ ignores the integers below $\sqrt x$, and overestimates later because divisibility by large primes is not independent. The plus product divided by $\log x$ approaches $\frac{6e^\gamma}{\pi^2}=1.0828$ from above.

### P1.4

In [ ]:
# CS application: trial division before Miller-Rabin in prime generation.
def miller_rabin(n, rounds=20):
    if n < 4:
        return n in (2, 3)
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2; s += 1
    for _ in range(rounds):
        a = random.randrange(2, n - 1)
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True

random.seed(1)
BITS = 1024
small = [int(p) for p in primes[1:] if p < 10**5]   # odd primes below 1e5
def survives(n, bound):
    for p in small:
        if p > bound:
            return True
        if n % p == 0:
            return False
    return True

print(" bound B   survivors (measured)   prod_{3<=p<=B}(1-1/p)   2e^-gamma/log B")
cands = [random.getrandbits(BITS) | (1 << (BITS - 1)) | 1 for _ in range(100000)]
for bound in (10, 100, 1000, 10000):
    frac = sum(survives(n, bound) for n in cands) / len(cands)
    i = pi_of(bound) - 1
    pred = math.exp(cum_log1m[i]) / 0.5
    print(f"  {bound:6d}      {frac:.4f}                  {pred:.4f}                 {2 * math.exp(-gamma) / math.log(bound):.4f}")

In [ ]:
def gen_prime(bits, bound):
    tests = 0
    while True:
        n = random.getrandbits(bits) | (1 << (bits - 1)) | 1
        if bound and not survives(n, bound):
            continue
        tests += 1
        if miller_rabin(n, rounds=1) and miller_rabin(n):
            return n, tests

for bound in (0, 100, 2000, 20000):
    random.seed(7)
    t0 = time.perf_counter()
    tot = 0
    for _ in range(10):
        _, t = gen_prime(BITS, bound)
        tot += t
    dt = (time.perf_counter() - t0) / 10
    print(f"trial division up to {bound:6d}: {tot / 10:6.1f} Miller-Rabin calls per prime, {1000 * dt:7.1f} ms per prime")

A random odd number survives trial division by the odd primes up to $B$ with probability $\prod_{3\le p\le B}(1-\frac1p)=2\prod_{p\le B}(1-\frac1p)\approx\frac{2e^{-\gamma}}{\log B}$: the measured fractions match the product to sampling error ($\pm0.002$), and Mertens's asymptotic is within about $1\%$ from $B=100$ on. Without sieving, about $\frac{\log2^{1024}}2\approx355$ odd candidates are needed per prime (our ten primes needed $475$ on average, a noisy geometric count); sieving to $B=20\,000$ brings the Miller–Rabin calls down to $37$ and the time per prime down by a factor of about $8$. The gain flattens because the survival fraction decreases only like $\frac1{\log B}$ while trial division costs grow like $\frac B{\log B}$: doubling $B$ saves a factor of only $\frac{\log B}{\log 2B}$. Real libraries (OpenSSL) sieve with a few thousand small primes, and update the residues incrementally instead of dividing each candidate afresh.